# EKG Processing Code- Testing


List of the additional features needed for me to add into the code:

- Data labelling optimized for data pipeline
    - This will help with creating a dataloader to incorporate the data into a pipeline.
- Consider addressing the biology in the coding decisions
    - This will have a direct impact on the feature engineering.
- Need to recreate neurokit2 in order to be more conducive to peak isolation as well as addressing the drift issue.
- Develop scalability for DevOps to create an applet


## Import Necessary Packages

In [ ]:
# pip install neurokit2
import json
import numpy as np
import matplotlib.pyplot as plt
import neurokit2 as nk # Source https://neuropsychology.github.io/NeuroKit/functions/ecg.html
import pandas as pd
import matplotlib.pyplot as plt

## Data PreProcessing

### JSON -> Python Friendly Format

In [ ]:
# The ECG file is stored in a computer-readable format called JSON, which is like a digital spreadsheet containing numbers and settings.


# Note to self- filename will be automated later on since multiple files will be processed
filename = "tache_lacasse_1min.txt"  # <- sample data

# filename = "CFA4_1min_standard_position.json" # <- normal data
# filename = "CFA2_1min_standard_position.json" # <- abnormal data
# filename = "CFA12_1min_standard_position.json" # <- borderline data


# === Load JSON ECG data ===
with open(filename, "r") as f:
    ecg_json = json.load(f)

### Extracting Key Data from the JSON

In [ ]:
# Here we’re picking out the key facts from the JSON file's structure:

# Some of this data includes:

# ecg_data = all the raw electrical readings from the electrodes (the heartbeat signals).
# fs = how many times per second the signal was measured (the sampling rate, here 500 times per second).
# scale = how to convert the raw numbers into meaningful electrical units (millivolts).
# num_leads = how many electrodes were used (8 in this case).

# === Extract data based on the header info ===
ecg_data = np.array(ecg_json["ecg"])
fs = ecg_json["header"]["samplingRate"]          # Sampling frequency (Hz)
scale = ecg_json["header"]["unitsPerMv"]         # Conversion factor (counts per mV)
num_leads = ecg_json["footer"]["numberOfLeads"]  # In our case, this is 8
duration_ms = ecg_json["footer"]["duration"]     # Duration in ms


# Based on the values for fs (sampling frequency), we know that the ECG machine records one number every 1/500th of a second.
# This block of code builds a timeline that tells us exactly when each heartbeat point occurred and creates a corresponding time vector.

# === Prepare time vector ===
n_samples = ecg_data.shape[0]
t = np.arange(n_samples) / fs  # seconds

### Rescale the data

In [ ]:
# The machine originally saves numbers that represent electrical voltage, but not in “real world” units.
# Dividing by the "scale" value turns them into millivolts (mV), which is the standard medical unit for ECG amplitude.
# === Convert raw data to mV ===
ecg_mV = ecg_data / scale

### Plot the current data


In [ ]:
# ecg_mV = ecg_data
# === Plot each lead separately ===
for i in range(num_leads):
    plt.figure(figsize=(12, 4))
    plt.plot(t, ecg_mV[:, i], linewidth=1)
    plt.title(f"Lead {i+1}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (mV)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

### Rescale the data to be in milivolts

In [ ]:
# Convert raw counts -> millivolts
ecg_mv = ecg_data / scale

### -and then begin to separate each lead into unique vectors

In [ ]:
# Original 8 leads (by index)
vec0 = ecg_mv[:, 0]  # Lead I
vec1 = ecg_mv[:, 1]  # Lead II
vec2 = ecg_mv[:, 2]  # V1
vec3 = ecg_mv[:, 3]  # V2
vec4 = ecg_mv[:, 4]  # V3
vec5 = ecg_mv[:, 5]  # V4
vec6 = ecg_mv[:, 6]  # V5
vec7 = ecg_mv[:, 7]  # V6

### -and therein use these separate leads to create a limb-by-limb representation of the leads using interpretable variable names.


In [ ]:
# Limb leads
lead_I   = vec0
lead_II  = vec1
lead_III = vec1 - vec0

# Augmented limb leads
lead_aVR = -(vec0 + vec1) / 2
lead_aVL = vec0 - vec1 / 2
lead_aVF = vec1 - vec0 / 2

# Precordial leads
lead_V1 = vec2
lead_V2 = vec3
lead_V3 = vec4
lead_V4 = vec5
lead_V5 = vec6
lead_V6 = vec7

### Now turn these leads into a dictionary

In [ ]:
# 12-lead ECG dictionary
ecg_12lead = {
    "I":   lead_I,
    "II":  lead_II,
    "III": lead_III,
    "aVR": lead_aVR,
    "aVL": lead_aVL,
    "aVF": lead_aVF,
    "V1":  lead_V1,
    "V2":  lead_V2,
    "V3":  lead_V3,
    "V4":  lead_V4,
    "V5":  lead_V5,
    "V6":  lead_V6,
}


# Store this data into a giant DataFrame
ecg_df = pd.DataFrame(ecg_12lead)
ecg_df["time"] = t

### Using NeuroKit to Start Basic Feature Engineering

#### Source Code:

https://neuropsychology.github.io/NeuroKit/functions/ecg.html

In [ ]:
# Process each lead independently, but keep everything in a dictionary.
def process_ecg_lead(ecg_signal, fs):
    """
    Run NeuroKit ECG processing on a single lead.
    Returns signals DataFrame and info dict.
    """
    signals, info = nk.ecg_process(ecg_signal, sampling_rate=fs)
    return signals, info


In [ ]:
# Build labeled signal DataFrame (P, Q, R, S, T)
def build_labeled_signal(signals):
    labels = np.full(len(signals), '', dtype=object)

    peak_columns = {
        'P': 'ECG_P_Peaks',
        'Q': 'ECG_Q_Peaks',
        'R': 'ECG_R_Peaks',
        'S': 'ECG_S_Peaks',
        'T': 'ECG_T_Peaks'
    }

    for label, col in peak_columns.items():
        if col in signals:
            labels[signals[col] == 1] = label

    df = pd.DataFrame({
        'ECG_Raw': signals['ECG_Raw'],
        'ECG_Clean': signals['ECG_Clean'],
        'Label': labels
    })

    return df


In [ ]:
# Segment beats individually using P-peaks
def segment_by_r_peaks(labeled_df):
    r_indices = labeled_df.index[labeled_df['Label'] == 'P'].tolist()

    segments = []
    for i in range(len(r_indices) - 1):
        seg = labeled_df.loc[r_indices[i]:r_indices[i+1]].copy()
        segments.append(seg)

    return segments


## Core Feature Engineering Algorithm


Note that I can add to this function the capability to extract further features


In [ ]:
#### DEFUNCT FEATURE ENGINEERING ALGORITHM TO ENSUSE THAT OVERWRITTEN T IS USED

# def extract_features_from_segment(segment, fs):
#     features = {}

#     # ---------------------------------------------
#     # Locate peaks relying on neurokit labels
#     # ---------------------------------------------
#     peaks = {}
#     for label in ['P', 'Q', 'R', 'S', 'T']:
#         idxs = segment.index[segment['Label'] == label].tolist()
#         peaks[label] = idxs[0] if idxs else np.nan

#     # ---------------------------------------------------------------------
#     # T-wave fallback logic- estimate T if missing: GPT defined function
#     # ---------------------------------------------------------------------

#     t_is_estimated = False

#     if np.isnan(peaks['T']) and not np.isnan(peaks['S']):
#         # Define search window: S peak → end of beat
#         start_idx = int(peaks['S'])
#         end_idx = segment.index[-1]

#         window = segment.loc[start_idx:end_idx, 'ECG_Clean']

#         if len(window) > 0:
#             # T peak = max absolute deflection
#             t_relative_idx = np.argmax(np.abs(window.values))
#             peaks['T'] = window.index[t_relative_idx]
#             t_is_estimated = True

#     features["T_estimated"] = int(t_is_estimated)

#     # -----------------------------
#     #  Engineer Interval Length Features
#     # -----------------------------
#     def time_diff(a, b):
#         if np.isnan(a) or np.isnan(b):
#             return np.nan
#         return abs(b - a) / fs

#     features["P_Q"] = time_diff(peaks['P'], peaks['Q'])
#     features["Q_R"] = time_diff(peaks['Q'], peaks['R'])
#     features["R_S"] = time_diff(peaks['R'], peaks['S'])
#     features["S_T"] = time_diff(peaks['S'], peaks['T'])
#     features["P_R"] = time_diff(peaks['P'], peaks['R'])


#     # -----------------------------
#     # Slope features
#     # -----------------------------
#     def slope(idx1, idx2):
#         if np.isnan(idx1) or np.isnan(idx2):
#             return np.nan
#         y1 = segment.loc[idx1, 'ECG_Clean']
#         y2 = segment.loc[idx2, 'ECG_Clean']
#         return (y2 - y1) / ((idx2 - idx1) / fs)

#     features["QR_slope"] = slope(peaks['Q'], peaks['R'])
#     features["RS_slope"] = slope(peaks['R'], peaks['S'])

#     # -----------------------------
#     # Area under curve features (also not yet working)
#     # -----------------------------
#     signal = segment['ECG_Clean'].values
#     features["AUC_full"] = np.trapz(np.abs(signal), dx=1/fs)

#     if not np.isnan(peaks['Q']) and not np.isnan(peaks['S']):
#         qrs = segment.loc[peaks['Q']:peaks['S'], 'ECG_Clean']
#         features["AUC_QRS"] = np.trapz(np.abs(qrs), dx=1/fs)
#     else:
#         features["AUC_QRS"] = np.nan

#     return features





In [ ]:
def extract_features_from_segment(segment, fs):
    features = {}

    # ---------------------------------------------
    # Locate peaks relying on NeuroKit labels
    # ---------------------------------------------
    peaks = {}
    for label in ['T','P', 'Q', 'R', 'S', 'T']:
        idxs = segment.index[segment['Label'] == label].tolist()
        peaks[label] = idxs[0] if idxs else np.nan

    # ---------------------------------------------------------------------
    # T-wave fallback logic: estimate T if missing
    # ---------------------------------------------------------------------
    t_is_estimated = False

    if np.isnan(peaks['T']) and not np.isnan(peaks['S']):
        start_idx = int(peaks['S'])
        end_idx = segment.index[-1]

        window = segment.loc[start_idx:end_idx, 'ECG_Clean']

        if len(window) > 0:
            # Estimate T as max absolute deflection after S
            t_relative_idx = np.argmax(np.abs(window.values))
            estimated_T_idx = window.index[t_relative_idx]

            # ---- CRITICAL FIX ----
            peaks['T'] = estimated_T_idx
            segment.loc[estimated_T_idx, 'Label'] = 'T'
            t_is_estimated = True

    features["T_estimated"] = int(t_is_estimated)
    
    # Try
    # 


    # -----------------------------
    # Interval length features
    # -----------------------------
    def time_diff(a, b):
        if np.isnan(a) or np.isnan(b):
            return np.nan
        return abs(b - a) / fs

    features["P_Q"] = time_diff(peaks['P'], peaks['Q'])
    features["Q_R"] = time_diff(peaks['Q'], peaks['R'])
    features["R_S"] = time_diff(peaks['R'], peaks['S'])
    features["S_T"] = time_diff(peaks['S'], peaks['T'])
    features["P_R"] = time_diff(peaks['P'], peaks['R'])

    # -----------------------------
    # Slope features
    # -----------------------------
    def slope(idx1, idx2):
        if np.isnan(idx1) or np.isnan(idx2):
            return np.nan
        y1 = segment.loc[idx1, 'ECG_Clean']
        y2 = segment.loc[idx2, 'ECG_Clean']
        return (y2 - y1) / ((idx2 - idx1) / fs)

    features["QR_slope"] = slope(peaks['Q'], peaks['R'])
    features["RS_slope"] = slope(peaks['R'], peaks['S'])

    # -----------------------------
    # Area under curve features
    # -----------------------------
    signal = segment['ECG_Clean'].values
    features["AUC_full"] = np.trapz(np.abs(signal), dx=1/fs)

    if not np.isnan(peaks['Q']) and not np.isnan(peaks['S']):
        qrs = segment.loc[peaks['Q']:peaks['S'], 'ECG_Clean']
        features["AUC_QRS"] = np.trapz(np.abs(qrs), dx=1/fs)
    else:
        features["AUC_QRS"] = np.nan

    return features


In [ ]:
# Improved extract_features_from_segment: ensures fallback T logic is used for all calculations

def extract_features_from_segment(segment, fs):
    features = {}

    # ---------------------------------------------
    # Locate peaks relying on NeuroKit labels
    # ---------------------------------------------
    peaks = {}
    for label in ['T', 'P', 'Q', 'R', 'S', 'T']:
        idxs = segment.index[segment['Label'] == label].tolist()
        peaks[label] = idxs[0] if idxs else np.nan

    # ---------------------------------------------------------------------
    # T-wave fallback logic: estimate T if missing
    # ---------------------------------------------------------------------
    t_is_estimated = False

    if np.isnan(peaks['T']) and not np.isnan(peaks['S']):
        start_idx = int(peaks['S'])
        end_idx = segment.index[-1]

        window = segment.loc[start_idx:end_idx, 'ECG_Clean']

        if len(window) > 0:
            t_relative_idx = np.argmax(np.abs(window.values))
            estimated_T_idx = window.index[t_relative_idx]
            peaks['T'] = estimated_T_idx
            segment.loc[estimated_T_idx, 'Label'] = 'T'
            t_is_estimated = True

    features["T_estimated"] = int(t_is_estimated)

    # All subsequent calculations use the (possibly newly assigned) T value
    def time_diff(a, b):
        if np.isnan(a) or np.isnan(b):
            return np.nan
        return abs(b - a) / fs

    features["P_Q"] = time_diff(peaks['P'], peaks['Q'])
    features["Q_R"] = time_diff(peaks['Q'], peaks['R'])
    features["R_S"] = time_diff(peaks['R'], peaks['S'])
    features["S_T"] = time_diff(peaks['S'], peaks['T'])
    features["P_R"] = time_diff(peaks['P'], peaks['R'])

    def slope(idx1, idx2):
        if np.isnan(idx1) or np.isnan(idx2):
            return np.nan
        y1 = segment.loc[idx1, 'ECG_Clean']
        y2 = segment.loc[idx2, 'ECG_Clean']
        return (y2 - y1) / ((idx2 - idx1) / fs)

    features["QR_slope"] = slope(peaks['Q'], peaks['R'])
    features["RS_slope"] = slope(peaks['R'], peaks['S'])

    signal = segment['ECG_Clean'].values
    features["AUC_full"] = np.trapz(np.abs(signal), dx=1/fs)

    if not np.isnan(peaks['Q']) and not np.isnan(peaks['S']):
        qrs = segment.loc[peaks['Q']:peaks['S'], 'ECG_Clean']
        features["AUC_QRS"] = np.trapz(np.abs(qrs), dx=1/fs)
    else:
        features["AUC_QRS"] = np.nan

    # Amplitude & voltage features
    def get_amplitude(label):
        if np.isnan(peaks[label]):
            return np.nan
        return segment.loc[peaks[label], 'ECG_Clean']

    features["R_amplitude"] = get_amplitude('R')
    features["S_amplitude"] = get_amplitude('S')
    features["P_amplitude"] = get_amplitude('P')
    features["T_amplitude"] = get_amplitude('T')
    features["Q_amplitude"] = get_amplitude('Q')

    baseline = np.mean([signal[0], signal[-1]])
    features["ST_baseline_diff"] = (get_amplitude('T') - baseline) if not np.isnan(peaks['T']) else np.nan
    features["P_baseline_diff"] = (get_amplitude('P') - baseline) if not np.isnan(peaks['P']) else np.nan

    return features

In [ ]:
# # Add amplitude and voltage features to extract_features_from_segment

# def extract_features_from_segment(segment, fs):
#     features = {}

#     # ---------------------------------------------
#     # Locate peaks relying on NeuroKit labels
#     # ---------------------------------------------
#     peaks = {}
#     for label in ['T','P', 'Q', 'R', 'S', 'T']:
#         idxs = segment.index[segment['Label'] == label].tolist()
#         peaks[label] = idxs[0] if idxs else np.nan

#     # ---------------------------------------------------------------------
#     # T-wave fallback logic: estimate T if missing
#     # ---------------------------------------------------------------------
#     t_is_estimated = False

#     if np.isnan(peaks['T']) and not np.isnan(peaks['S']):
#         start_idx = int(peaks['S'])
#         end_idx = segment.index[-1]

#         window = segment.loc[start_idx:end_idx, 'ECG_Clean']

#         if len(window) > 0:
#             t_relative_idx = np.argmax(np.abs(window.values))
#             estimated_T_idx = window.index[t_relative_idx]
#             peaks['T'] = estimated_T_idx
#             segment.loc[estimated_T_idx, 'Label'] = 'T'
#             t_is_estimated = True

#     features["T_estimated"] = int(t_is_estimated)

#     # -----------------------------
#     # Interval length features
#     # -----------------------------
#     def time_diff(a, b):
#         if np.isnan(a) or np.isnan(b):
#             return np.nan
#         return abs(b - a) / fs

#     features["P_Q"] = time_diff(peaks['P'], peaks['Q'])
#     features["Q_R"] = time_diff(peaks['Q'], peaks['R'])
#     features["R_S"] = time_diff(peaks['R'], peaks['S'])
#     features["S_T"] = time_diff(peaks['S'], peaks['T'])
#     features["P_R"] = time_diff(peaks['P'], peaks['R'])

#     # -----------------------------
#     # Slope features
#     # -----------------------------
#     def slope(idx1, idx2):
#         if np.isnan(idx1) or np.isnan(idx2):
#             return np.nan
#         y1 = segment.loc[idx1, 'ECG_Clean']
#         y2 = segment.loc[idx2, 'ECG_Clean']
#         return (y2 - y1) / ((idx2 - idx1) / fs)

#     features["QR_slope"] = slope(peaks['Q'], peaks['R'])
#     features["RS_slope"] = slope(peaks['R'], peaks['S'])

#     # -----------------------------
#     # Area under curve features
#     # -----------------------------
#     signal = segment['ECG_Clean'].values
#     features["AUC_full"] = np.trapz(np.abs(signal), dx=1/fs)

#     if not np.isnan(peaks['Q']) and not np.isnan(peaks['S']):
#         qrs = segment.loc[peaks['Q']:peaks['S'], 'ECG_Clean']
#         features["AUC_QRS"] = np.trapz(np.abs(qrs), dx=1/fs)
#     else:
#         features["AUC_QRS"] = np.nan

#     # ===== NEW: Amplitude & Voltage Features =====
#     # Peak amplitude values
#     def get_amplitude(label):
#         if np.isnan(peaks[label]):
#             return np.nan


#         return segment.loc[peaks[label], 'ECG_Clean']

#     features["R_amplitude"] = get_amplitude('R')
#     features["S_amplitude"] = get_amplitude('S')
#     features["P_amplitude"] = get_amplitude('P')
#     features["T_amplitude"] = get_amplitude('T')
#     features["Q_amplitude"] = get_amplitude('Q')

#     # Baseline voltage (average of segment start/end)
#     baseline = np.mean([signal[0], signal[-1]])
#     features["ST_baseline_diff"] = (get_amplitude('T') - baseline) if not np.isnan(peaks['T']) else np.nan
#     features["P_baseline_diff"] = (get_amplitude('P') - baseline) if not np.isnan(peaks['P']) else np.nan

#     return features

In [ ]:
# # Process all leads x all beats
# all_features = []

# for lead_name, ecg_signal in ecg_12lead.items():
#     print(f"Processing lead {lead_name}")

#     signals, info = process_ecg_lead(ecg_signal, fs)
#     labeled_df = build_labeled_signal(signals)
#     segments = segment_by_r_peaks(labeled_df)

#     for beat_idx, segment in enumerate(segments):
#         feat = extract_features_from_segment(segment, fs)
#         feat["Lead"] = lead_name
#         feat["Beat"] = beat_idx
#         all_features.append(feat)



all_features = []

for lead_name, ecg_signal in ecg_12lead.items():
    print(f"Processing lead {lead_name}")

    signals, info = process_ecg_lead(ecg_signal, fs)
    labeled_df = build_labeled_signal(signals)
    segments = segment_by_r_peaks(labeled_df)

    for beat_idx, segment in enumerate(segments):
        feat = extract_features_from_segment(segment, fs)

        # ---- Metadata ----
        feat["Lead"] = lead_name
        feat["Beat_Index"] = beat_idx          # nth wave
        feat["File_Name"] = filename            # file-level ID
        feat["Num_Samples"] = len(segment)      # optional but useful

        all_features.append(feat)


In [ ]:
# Final ML-ready feature table

features_df = pd.DataFrame(all_features)

# # Optional aggregation (mean/std per lead per recording)
# aggregated = (
#     features_df
#     .groupby("Lead")
#     .agg(["mean", "std", "median"])
# )

In [ ]:
features_df = pd.DataFrame(all_features)

print(features_df.head(10))



# Features to Add

- Put back the plotting to show how often P and T are mentioned
- Put in fail safe for P-wave identification

In [ ]:
def plot_segment_with_peaks(segment, fs, lead, beat_idx, filename):
    """
    Plot one ECG segment with PQRST labels.
    """
    time = np.arange(len(segment)) / fs
    signal = segment['ECG_Clean'].values

    plt.figure(figsize=(12, 4))
    plt.plot(time, signal, linewidth=1)

    # Peak markers
    peak_colors = {
        'P': 'green',
        'Q': 'blue',
        'R': 'red',
        'S': 'orange',
        'T': 'purple'
    }

    for label, color in peak_colors.items():
        idxs = segment.index[segment['Label'] == label].tolist()
        if idxs:
            local_idx = segment.index.get_indexer(idxs)
            plt.scatter(
                time[local_idx],
                signal[local_idx],
                color=color,
                label=label,
                zorder=3
            )

    plt.title(f"File: {filename} | Lead: {lead} | Beat: {beat_idx}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (mV)")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
def print_feature_row(features_df, filename, lead, beat_idx):
    row = features_df[
        (features_df["File_Name"] == filename) &
        (features_df["Lead"] == lead) &
        (features_df["Beat_Index"] == beat_idx)
    ]

    if row.empty:
        print("⚠️ No feature row found.")
    else:
        print("Feature values:")
        display(row.reset_index(drop=True))


In [ ]:
for lead_name, ecg_signal in ecg_12lead.items():
    print(f"\n==============================")
    print(f"Processing Lead: {lead_name}")
    print(f"==============================")

    signals, info = process_ecg_lead(ecg_signal, fs)
    labeled_df = build_labeled_signal(signals)
    segments = segment_by_r_peaks(labeled_df)

    for beat_idx, segment in enumerate(segments):
        print(f"\n--- File: {filename} | Lead: {lead_name} | Beat: {beat_idx} ---")

        # 1. Plot ECG with labels
        plot_segment_with_peaks(
            segment=segment,
            fs=fs,
            lead=lead_name,
            beat_idx=beat_idx,
            filename=filename
        )

        # 2. Print corresponding feature row
        print_feature_row(
            features_df=features_df,
            filename=filename,
            lead=lead_name,
            beat_idx=beat_idx
        )

        # Optional pause (uncomment if needed)
        # input("Press Enter to continue...") 
        
        
# Put subplots to plot all 12 leads at one time
# Incorporate a time counter to mention the universal start and stop time for each beat to
# make it easier to remove them based on physical annotations


In [ ]:
features_df

In [ ]:
features_df.describe()

In [ ]:
# Export features to xlsx for ML modeling
features_df.to_excel("ecg_features.xlsx", index=False)